# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [ ]:
# Load the libraries as required.
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
import pickle

In [ ]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


In [ ]:
fires_dt.head()

# Get X and Y

Create the features data frame and target data.

In [ ]:
X = fires_dt.drop(columns='area')
y = fires_dt['area']

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [ ]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

X_num = X[numeric_features]
X_cat = X[categorical_features]

transformer1 = ColumnTransformer(
    transformers=[
        ('num_transform', StandardScaler(), X_num.columns.values),
        ('cat_transform', OneHotEncoder(handle_unknown='infrequent_if_exist', drop='if_binary'), X_cat.columns.values),
    ],
    remainder='drop'
)

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [ ]:
numeric_transformer2 = Pipeline([
    ('Poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler())
])
transformer2 = ColumnTransformer(
    transformers=[       
        ('num_transform', numeric_transformer2, X_num.columns.values ),
        ('cat_transform', OneHotEncoder(handle_unknown='infrequent_if_exist', drop='if_binary'), X_cat.columns.values), 
    ], remainder='drop'
)

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [ ]:
# Pipeline A = preproc1 + baseline

pipeA = Pipeline(
    [
        ('preproc1', transformer1), 
        ('Ridge', Ridge())
    ]
)
pipeA

In [ ]:
# Pipeline B = preproc2 + baseline
pipeB = Pipeline(
    [
        ('preproc2', transformer2), 
        ('Knn', KNeighborsRegressor())
    ]
)
pipeB

In [ ]:
# Pipeline C = preproc1 + advanced model
pipeC = Pipeline(
    [
        ('preproc1', transformer1), 
        ('RF_Regressor', RandomForestRegressor(n_jobs=-1, random_state=42))
    ]
)
pipeC

In [ ]:
# Pipeline D = preproc2 + advanced model
pipeD = Pipeline(
    [
        ('preproc2', transformer2), 
        ('HistGradient_BR', HistGradientBoostingRegressor(random_state=42))
    ]
)
pipeD
    

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [ ]:
param_grid_ridge = {
    'Ridge__alpha': [0.1, 1.0, 10.0],
    'Ridge__solver': ['auto', 'svd', 'cholesky', 'lsqr']
}
scoring = ['neg_root_mean_squared_error', 'neg_mean_absolute_error']
grid_cv1 = GridSearchCV(
    estimator=pipeA,
    param_grid=param_grid_ridge,
    scoring=scoring,
    cv=5,
    refit='neg_root_mean_squared_error'
)
grid_cv1.fit(X_train, Y_train)

results_df = pd.DataFrame(grid_cv1.cv_results_)
results_df[['params', 'mean_test_neg_root_mean_squared_error', 'mean_test_neg_mean_absolute_error']].sort_values('mean_test_neg_root_mean_squared_error', ascending=False)


In [ ]:
param_grid_k_neighbors = {
    'Knn__n_neighbors': [3, 5, 6],
    'Knn__weights': ['uniform', 'distance']
}
scoring = ['neg_root_mean_squared_error', 'neg_mean_absolute_error']
grid_cv2 = GridSearchCV(
    estimator=pipeB,
    param_grid=param_grid_k_neighbors,
    scoring=scoring,
    cv=5,
    refit='neg_root_mean_squared_error'
)
grid_cv2.fit(X_train, Y_train)
results_df = pd.DataFrame(grid_cv2.cv_results_)
results_df[['params', 'mean_test_neg_root_mean_squared_error', 'mean_test_neg_mean_absolute_error']].sort_values('mean_test_neg_root_mean_squared_error', ascending=False)

In [ ]:
param_grid_rf = {
    "RF_Regressor__n_estimators": [100, 300, 500],
    "RF_Regressor__max_depth": [5, 10],
    "RF_Regressor__min_samples_split": [2, 5],
    "RF_Regressor__min_samples_leaf": [1, 2],
}
scoring = ['neg_root_mean_squared_error', 'neg_mean_absolute_error']
grid_cv3 = GridSearchCV(
    estimator=pipeC,
    param_grid=param_grid_rf,
    scoring=scoring,
    cv=5,
    refit='neg_root_mean_squared_error',
    n_jobs=-1
)
grid_cv3.fit(X_train, Y_train)
results_df = pd.DataFrame(grid_cv3.cv_results_)
results_df[['params', 'mean_test_neg_root_mean_squared_error', 'mean_test_neg_mean_absolute_error']].sort_values('mean_test_neg_root_mean_squared_error', ascending=False)

In [ ]:
param_grid_histgb = {
    "HistGradient_BR__learning_rate": [0.01, 0.05, 0.1],
    "HistGradient_BR__max_depth": [3, 5, 7],
    "HistGradient_BR__max_iter": [100, 300],
    "HistGradient_BR__min_samples_leaf": [10, 20],
    "HistGradient_BR__max_leaf_nodes": [15, 31, 63]
}
scoring = ['neg_root_mean_squared_error', 'neg_mean_absolute_error']
grid_cv4 = GridSearchCV(
    estimator=pipeD,
    param_grid=param_grid_histgb,
    scoring=scoring,
    cv=5,
    refit='neg_root_mean_squared_error',
)
grid_cv4.fit(X_train, Y_train)
results_df = pd.DataFrame(grid_cv4.cv_results_)
results_df[['params', 'mean_test_neg_root_mean_squared_error', 'mean_test_neg_mean_absolute_error']].sort_values('mean_test_neg_root_mean_squared_error', ascending=False)

# Evaluate

+ Which model has the best performance?

# Export

+ Save the best performing model to a pickle file.

In [ ]:
best_rmse_cv1 = -grid_cv1.best_score_
best_rmse_cv1

In [ ]:
best_rmse_cv2 = -grid_cv2.best_score_
best_rmse_cv2

In [ ]:
best_rmse_cv3 = -grid_cv3.best_score_
best_rmse_cv3

In [ ]:
best_rmse_cv4 = -grid_cv4.best_score_
best_rmse_cv4

In [ ]:
model_results = pd.DataFrame({
    'Model': [
        'Ridge Regression',
        'KNN Regressor',
        'Random Forest Regressor',
        'HistGradient Boosting Regressor',
    ],
    'best_cv_rmse': [
        -grid_cv1.best_score_,
        -grid_cv2.best_score_,
        -grid_cv3.best_score_,
        -grid_cv4.best_score_,
    ],
    'Best_params': [
        str(grid_cv1.best_params_),
        str(grid_cv2.best_params_),
        str(grid_cv3.best_params_),
        str(grid_cv4.best_params_),
    ],
})
model_results = model_results.sort_values('best_cv_rmse', ignore_index=True)
model_results

In [ ]:
best_model = grid_cv4.best_estimator_
y_pred = best_model.predict(X_test)
test_rmse = root_mean_squared_error(Y_test, y_pred)
test_mae = mean_absolute_error(Y_test, y_pred)
print(f"Test RMSE: {test_rmse}")
print(f"Test MAE: {test_mae}")

In [ ]:
best_model = grid_cv4.best_estimator_
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [ ]:
import shap

preprocessor = best_model.named_steps['preproc2']
regressor = best_model.named_steps['HistGradient_BR']

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

explainer = shap.Explainer(
    regressor, 
    X_train_transformed, 
    feature_names=feature_names
)
shap_values = explainer(X_test_transformed, check_additivity=False)

In [ ]:
shap.plots.waterfall(shap_values[0], max_display=10)

In [ ]:
shap.plots.beeswarm(shap_values, max_display=20)

*(Answer here.)*

The local SHAP waterfall plot explains the prediction for one test observation. Features with positive SHAP values (like cat_transform__day_sat) pushed the predicted burned area higher, while features with negative SHAP values (like num_transform__coord_x temp) pushed the prediction lower.

The global SHAP beeswarm plot shows which features were most influential across the test set. The most important variables (like num_transform__coord_x temp, num_transform__dmc rh, num_transform__dmc temp, num_transform__coord_y temp, num_transform__temp wind, and num_transform__coord_y dc) appear at the top of the plot because they have the largest average absolute SHAP values.

Features with consistently low SHAP values (like num_transform__isi, num_transform_coord_y wind, or num_transform__dmc, num_transform__dc isi, num_transform___coord_x rh, and 65 other features) appear to contribute little to the model. These would be candidates for removal. To test whether removing them improves or hurts performance, I would retrain the same pipeline without those features and compare cross-validated RMSE and MAE against the original model.

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [x] Created a branch with the correct naming convention.
- [x] Ensured that the repository is public.
- [x] Reviewed the PR description guidelines and adhered to them.
- [x] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.